## Initializing the NUTS-HMC sampler

In this example, we show how to use parameter estimates returned by any of Stan's inference algorithms as initial values for Stan's NUTS-HMC sampler. These include:

* [Pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) 
* [ADVI ](https://mc-stan.org/docs/cmdstan-guide/variational_config.html) 
* [Laplace](https://mc-stan.org/docs/cmdstan-guide/laplace_sample_config.html)
* [Optimization](https://mc-stan.org/docs/cmdstan-guide/optimize_config.html)
* [NUTS-HMC MCMC](https://mc-stan.org/docs/cmdstan-guide/mcmc_config.html)

By default, the NUTS-HMC sampler randomly initializes all (unconstrained) model parameters uniformly in the interval (-2, 2).  If this interval is far from the typical set of the posterior, initializing sampling from these approximation algorithms can speed up and improve adaptation.

### Model and data

The Stan model and data are taken from the [posteriordb package](https://github.com/stan-dev/posteriordb).

We use the [blr model](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/models/stan/blr.stan),
a Bayesian standard linear regression model with noninformative priors,
and its corresponding simulated dataset [sblri.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data/sblri.json.zip),
which was simulated via script [sblr.R](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/data/data-raw/sblr/sblr.R).
For convenience, this example assumes the posteriordb model and data are local, in files `blr.stan` and `sblri.json`.

In [1]:
import os
from cmdstanpy import CmdStanModel

stan_file = 'blr.stan' # basic linear regression
data_file = 'sblri.json' # simulated data

model = CmdStanModel(stan_file=stan_file)

print(model.code())

15:51:19 - cmdstanpy - INFO - compiling stan file /home/runner/work/cmdstanpy/cmdstanpy/docsrc/users-guide/examples/blr.stan to exe file /home/runner/work/cmdstanpy/cmdstanpy/docsrc/users-guide/examples/blr


15:51:26 - cmdstanpy - INFO - compiled model executable: /home/runner/work/cmdstanpy/cmdstanpy/docsrc/users-guide/examples/blr


data {
  int<lower=0> N;
  int<lower=0> D;
  matrix[N, D] X;
  vector[N] y;
}
parameters {
  vector[D] beta;
  real<lower=0> sigma;
}
model {
  // prior
  target += normal_lpdf(beta | 0, 10);
  target += normal_lpdf(sigma | 0, 10);
  // likelihood
  target += normal_lpdf(y | X * beta, sigma);
}




### Demonstration with Stan's `pathfinder` method

Initializing the sampler with estimates from any previous inference algorithm follows the same general usage pattern. First, we call the 
corresponding method on the `CmdStanModel` object. From the resulting fit, we call the `.create_inits()` 
method to construct a set of per-chain initializations for the model parameters. To make it explicit, 
we will walk through the process using the `pathfinder` method (which wraps the 
CmdStan [pathfinder ](https://mc-stan.org/docs/cmdstan-guide/pathfinder-config.html) method).

Pathfinder locates normal approximations to the target
density along a quasi-Newton optimization path, with local covariance
estimated using the negative inverse Hessian estimates produced by the
LBFGS optimizer. Pathfinder returns draws from the Gaussian approximation
with the lowest estimated Kullback-Leibler (KL) divergence to the true
posterior.
By default, CmdStanPy runs multi-path Pathfinder which returns an importance-resampled 
set of draws over the outputs of 4 independent single-path Pathfinders.
This better matches non-normal target densities and also mitigates
the problem of L-BFGS getting stuck at local optima or in saddle points on plateaus.

We obtain Pathfinder estimates by calling the `.pathfinder()` method which returns a `CmdStanPathfinder` object:

In [2]:
pathfinder_fit = model.pathfinder(data=data_file, seed=123)

15:51:26 - cmdstanpy - INFO - Chain [1] start processing


15:51:26 - cmdstanpy - INFO - Chain [1] done processing


Posteriordb provides reference posteriors for all models. For the blr model, conditioned on the dataset `sblri.json`, the reference posteriors can be found in the [sblri-blr.json](https://github.com/stan-dev/posteriordb/blob/master/posterior_database/reference_posteriors/summary_statistics/mean/mean/sblri-blr.json) file.

The reference posteriors for all elements of `beta` and `sigma` are all very close to 1.0.

The experiments reported in Figure 3 of the paper [Pathfinder: Parallel quasi-Newton variational inference](https://arxiv.org/abs/2108.03782) by Zhang et al. show that Pathfinder provides a better estimate of the posterior, as measured by the 1-Wasserstein distance to the reference posterior, than 75 iterations of the warmup Phase I algorithm used by the NUTS-HMC sampler.
Furthermore, Pathfinder is more computationally efficient, requiring fewer evaluations of the log density and gradient functions. Therefore, using the Pathfinder estimates to initialize the parameter values for the NUTS-HMC sampler can allow the sampler to do a better job of adapting the stepsize and metric during warmup, resulting in better performance and estimation.

We construct the parameter inits for full MCMC sampling below. The `.create_inits()` default behavior is to create inits for four chains to correspond with the sampling defaults. You can requests more or less by modifying the `chains` keyword argument.

In [3]:
pathfinder_inits = pathfinder_fit.create_inits()
for chain_init in pathfinder_inits:
    print(chain_init)

{'beta': array([0.99990843, 1.0008725 , 0.99963515, 1.0012274 , 0.99842941]), 'sigma': array(0.90350659)}
{'beta': array([0.99855207, 1.0008112 , 0.99939398, 1.0005048 , 1.001184  ]), 'sigma': array(0.91757312)}
{'beta': array([1.0001093 , 0.99804565, 0.99956504, 0.9998398 , 1.0007671 ]), 'sigma': array(0.94063436)}
{'beta': array([1.000643  , 1.002155  , 0.99978113, 1.0030084 , 1.001805  ]), 'sigma': array(0.9846326)}


We see that the Pathfinder inits are close the reference posteriors for the parameters. To use these inits, we pass the `pathfinder_inits` object to the `inits` kwarg:

In [4]:
mcmc_pathfinder_inits_fit = model.sample(
    data=data_file, inits=pathfinder_inits, iter_warmup=75, seed=12345
)

15:51:26 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:26 - cmdstanpy - INFO - CmdStan done processing.


In [5]:
print(mcmc_pathfinder_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



Despite only running 75 warmup iterations, all posterior diagnostics from the sampler look good.

In [6]:
mcmc_pathfinder_inits_fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-156.868000,0.061286,1.775720,1.654450,-160.223000,-156.540000,-154.61200,929.341,1110.790,3171.810,1.002410
beta[1],0.999445,0.000014,0.000941,0.000931,0.997903,0.999433,1.00103,4674.320,3369.720,15953.300,1.000530
beta[2],1.000260,0.000017,0.001144,0.001104,0.998346,1.000260,1.00213,4333.350,3141.050,14789.600,1.001010
beta[3],1.000440,0.000013,0.000963,0.000943,0.998830,1.000450,1.00199,5293.810,3541.900,18067.600,0.999846
beta[4],1.001110,0.000016,0.001048,0.001066,0.999401,1.001110,1.00281,4464.080,3296.140,15235.800,1.001050
beta[5],1.001570,0.000016,0.001050,0.001059,0.999854,1.001580,1.00331,4440.810,2995.980,15156.300,1.003710
sigma,0.968202,0.004472,0.071634,0.071698,0.858751,0.965430,1.09381,249.760,238.609,852.424,1.017660


If we were to instead use the default random parameter initializations, we would need to run more warmup iterations to produce useful samples. For example, if we only run the same 75 warmup iterations with random inits, the result fails to estimate `sigma` correctly: 

In [7]:
mcmc_random_inits_fit = model.sample(data=data_file, iter_warmup=75, seed=12345)

15:51:26 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:26 - cmdstanpy - INFO - CmdStan done processing.


15:51:26 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 10 divergent transitions (1.0%)
	Chain 3 had 174 divergent transitions (17.4%)
	Chain 4 had 105 divergent transitions (10.5%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [8]:
mcmc_random_inits_fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-186.905000,24.818700,35.341700,29.698100,-246.822000,-176.50200,-154.91900,4.98442,2000.000,40.5237,2.53979
beta[1],0.999441,0.000113,0.002364,0.001368,0.995958,0.99946,1.00339,484.04200,2000.000,3935.3000,1.15098
beta[2],1.000090,0.000134,0.002730,0.001847,0.995429,1.00018,1.00449,426.47500,2000.000,3467.2800,1.28431
beta[3],1.000420,0.000106,0.002270,0.001447,0.996795,1.00042,1.00404,501.70500,2000.000,4078.9000,1.16093
beta[4],1.001340,0.000144,0.002320,0.001551,0.997944,1.00132,1.00501,295.28900,216.765,2400.7200,1.14912
beta[5],1.001500,0.000143,0.002671,0.001631,0.997447,1.00139,1.00522,499.25100,2000.000,4058.9500,1.18556
sigma,1.860840,0.737960,1.053270,0.858795,0.892552,1.55046,3.63793,4.95282,2000.000,40.2669,2.72262


In [9]:
print(mcmc_random_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
289 of 4000 (7.22%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
The E-BFMI, 0.01, is below the nominal threshold of 0.30 which suggests that HMC may have trouble exploring the target distribution.
If possible, try to reparameterize the model.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  beta[1], beta[2], beta[3], beta[4], beta[5], sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or 

The diagnostics clearly indicate problems with estimating `sigma`. In this case, it is necessary to run the model with at least 150 warmup iterations to produce a good set of estimates when starting from the default initialization.

### Other inference algorithms

We can follow the same pattern with Stan's ADVI algorithm by first using the `CmdStanModel.variationl` method. Because this algorithm is unstable and may fail to converge, we run it with argument `require_converged` set to `False`.  We also specify a seed, to avoid instabilities as well as for reproducibility.

In [10]:
vb_fit = model.variational(data=data_file, require_converged=False, seed=123)

15:51:27 - cmdstanpy - INFO - Chain [1] start processing


15:51:27 - cmdstanpy - INFO - Chain [1] done processing


15:51:27 - cmdstanpy - WARNING - The algorithm may not have converged.
Proceeding because require_converged is set to False


The ADVI algorithm provides estimates of all model parameters.

The `variational` method returns a `CmdStanVB` object, which similarly can construct a set of inits with the `.create_inits()` method:

In [11]:
vb_inits = vb_fit.create_inits()
for chain_init in vb_inits:
    print(chain_init)

{'beta': array([1.0040196 , 0.99813275, 1.0125375 , 0.99757411, 0.98988309]), 'sigma': array(1.6632041)}
{'beta': array([1.0057483 , 1.0001806 , 1.0077405 , 0.99796018, 0.99060487]), 'sigma': array(1.4770864)}
{'beta': array([1.0083486 , 0.995894  , 1.0103457 , 0.99940618, 0.98967933]), 'sigma': array(1.465479)}
{'beta': array([1.0084384 , 1.0034041 , 1.0104186 , 0.99850502, 0.99044621]), 'sigma': array(1.7089)}


Which can be passed to the `inits` keyword argument of the sample method.

In [12]:
mcmc_vb_inits_fit = model.sample(
    data=data_file, inits=vb_inits, iter_warmup=75, seed=12345
)

15:51:27 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:27 - cmdstanpy - INFO - CmdStan done processing.


15:51:27 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [13]:
print(mcmc_vb_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.



In [14]:
mcmc_vb_inits_fit.summary()

,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
lp__,-156.935000,0.059862,1.774190,1.626520,-160.300000,-156.596000,-154.71300,958.093,1923.540,3561.68,1.004990
beta[1],0.999436,0.000014,0.000986,0.000954,0.997803,0.999415,1.00104,5065.700,3456.690,18831.60,1.000180
beta[2],1.000240,0.000018,0.001166,0.001161,0.998319,1.000230,1.00216,4212.070,2932.800,15658.20,1.000830
beta[3],1.000440,0.000014,0.000962,0.000946,0.998851,1.000430,1.00203,5039.490,3605.040,18734.10,1.001040
beta[4],1.001170,0.000016,0.001069,0.001074,0.999425,1.001160,1.00294,4589.570,3309.640,17061.60,1.000150
beta[5],1.001540,0.000014,0.001030,0.001021,0.999874,1.001540,1.00323,5309.610,3035.620,19738.30,0.999665
sigma,0.970882,0.004057,0.072317,0.075456,0.851950,0.966701,1.09862,324.898,268.506,1207.80,1.008510


The sampler estimates match the reference posterior with no diagnostic issues.

Inits can also be constructed from the `laplace` method:

In [15]:
laplace_inits = model.laplace_sample(data=data_file, seed=123).create_inits()

15:51:27 - cmdstanpy - INFO - Chain [1] start processing


15:51:27 - cmdstanpy - INFO - Chain [1] done processing


15:51:27 - cmdstanpy - INFO - Chain [1] start processing


15:51:27 - cmdstanpy - INFO - Chain [1] done processing


In [16]:
mcmc_laplace_inits_fit = model.sample(
    data=data_file, inits=laplace_inits, iter_warmup=75, seed=12345
)

15:51:27 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:27 - cmdstanpy - INFO - CmdStan done processing.


In [17]:
print(mcmc_laplace_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



And the `optimize` method. Since optimizations attempts to return a posterior mode, this will initialize all chains at the same point, which is not typically ideal.

In [18]:
optimized_inits = model.optimize(data=data_file, seed=123).create_inits()

15:51:27 - cmdstanpy - INFO - Chain [1] start processing


15:51:27 - cmdstanpy - INFO - Chain [1] done processing


15:51:27 - cmdstanpy - WARNING - The default behavior of CmdStanMLE.stan_variable() will change in a future release to always return a numpy.ndarray, even for scalar variables.


In [19]:
mcmc_optimize_inits_fit = model.sample(
    data=data_file, inits=optimized_inits, iter_warmup=75, seed=12345
)

15:51:27 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:27 - cmdstanpy - INFO - CmdStan done processing.


In [20]:
print(mcmc_optimize_inits_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



It is also possible to use the output of the `sample()` method itself to construct inits to be fed into a future sampling run:

In [21]:
first_mcmc_inits = model.sample(
    data=data_file, iter_warmup=75, seed=12345
).create_inits()

15:51:27 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:27 - cmdstanpy - INFO - CmdStan done processing.


15:51:27 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 10 divergent transitions (1.0%)
	Chain 3 had 174 divergent transitions (17.4%)
	Chain 4 had 105 divergent transitions (10.5%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [22]:
second_mcmc_fit = model.sample(data=data_file, inits=first_mcmc_inits, iter_warmup=75, seed=12345)

15:51:27 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 3:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

chain 4:   0%|          | 0/1075 [00:00<?, ?it/s, (Warmup)]

15:51:27 - cmdstanpy - INFO - CmdStan done processing.


15:51:27 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Exception: normal_lpdf: Scale parameter is 0, but must be positive! (in 'blr.stan', line 16, column 2 to column 45)
Consider re-running with show_console=True if the above output is unclear!


In [23]:
print(second_mcmc_fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

The following parameters had rank-normalized split R-hat greater than 1.01:
  sigma
Such high values indicate incomplete mixing and biased estimation.
You should consider regularizing your model with additional prior information or a more effective parameterization.

Processing complete.



We see that despite the initial sampling issues in the first MCMC run, the inits sourced from that run result in reasonable sampling in the second run.